# Contextual Scouting — UEFA Euro 2024
## Hypothesis 2 (Contextual Decision Quality)

Aggregate pass completion rates lack analytical value when devoid of context. H2 evaluates **decision quality on the ball** by asking, for every open-play pass: *given the alternatives visible in the 360 freeze-frame, was the chosen pass the right one?*

The headline metric is **`DQ_ctx`**, the unweighted mean of three sub-indices (3 variables each, 9 in total):

| Sub-index | Question | Variables |
| --- | --- | --- |
| **CHOICE QUALITY** | did you pick well among alternatives? | `DQ_continuous`, `Top3 Rate`, `Regret /90` |
| **PRESSURE COMPOSURE** | does decision quality hold under pressure? | `ΔDQ_pressure`, `Completion @ high_press`, `Turnover Rate Under Pressure` |
| **VISION & AMBITION** | do you see and complete corridors others don't? | `Tight Corridor Rate`, `Tight Corridor Completion %`, `xEPV Premium /90` |

Each variable is converted to a **within-role percentile** (CB / FB / MID / CAM / WIDE / FW); each sub-index is the mean of its 3 percentiles; `DQ_ctx` is the mean of the three sub-indices.

The pipeline is built on top of two model components:

- **`xPass`** — a logistic regression returning P(complete | features), trained on 13 spatial / pressure / pass-type features extracted from the 360 freeze-frame at the moment of the pass. Crucially, features are computed **at the receiver's position in the frame**, not at `pass.end_location`, so the model can score *hypothetical* passes to teammates the player did not pick.
- **`xEPV`** for any (start → teammate) candidate: `xPass · EPV_post_pass + (1 − xPass) · EPV_turnover`, with `EPV_turnover ≈ −EPV(start_loc)` as a symmetric loss-of-possession proxy.

**Reusing H1 primitives.** All shared geometry stays in `src/geometry.py` of the H1 package: pressure radius (`PRESSURE_RADIUS = 2.5`), pressure threshold (`PRESSURE_MIN = 2`), corridor width (`CORRIDOR_M = 5`), pitch dimensions, role mapping. Same definition of *high pressure* across H1 and H2. New H2-specific code lives in the sibling package `Decision_Quality/dq/`.

**Pool filter.** Single threshold of **135 minutes played** — stricter than H1 (90/135 dual) because H2 metrics are per-pass and stabilise more slowly, especially on the high-pressure subset.


## Setup

Localizza la project root cercando la cartella sorella `Space_Control_and_Value` risalendo dal notebook. Aggiunge a `sys.path`:

- `<ROOT>/Space_Control_and_Value/` → permette `from src import config as h1_config` (riusa il package H1 senza duplicarlo).
- `<ROOT>/Decision_Quality/` → permette `from dq import ...` (package H2).

Da H1 importiamo direttamente: `config` (paths, soglie, role map), `geometry` (primitive in metri). Niente duplicazione di costanti.

In [1]:
import sys
from pathlib import Path

# Risali fino alla project root: la cartella che contiene Space_Control_and_Value/
NB_DIR = Path.cwd().resolve()
ROOT = NB_DIR
while ROOT.parent != ROOT and not (ROOT / 'Space_Control_and_Value').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'Space_Control_and_Value').is_dir():
    raise RuntimeError(f'project root non trovata risalendo da {NB_DIR}')

H1_DIR = ROOT / 'Space_Control_and_Value'
H2_DIR = ROOT / 'Decision_Quality'
for p in (str(H1_DIR), str(H2_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd

from src import config as h1_config
from src import geometry as geom

print(f'project root        : {ROOT}')
print(f'H1 package          : {H1_DIR}')
print(f'H2 package          : {H2_DIR}')
print(f'pressure radius     : {h1_config.PRESSURE_RADIUS} m  (>= {h1_config.PRESSURE_MIN} opp = high pressure)')
print(f'corridor width (5m) : {h1_config.CORRIDOR_M} m')
print(f'pool min minutes    : {h1_config.ANALYSIS_MIN_MINUTES} min')

project root        : /Users/matteovezzoli/Desktop/Contextual-Football-Scouting
H1 package          : /Users/matteovezzoli/Desktop/Contextual-Football-Scouting/Space_Control_and_Value
H2 package          : /Users/matteovezzoli/Desktop/Contextual-Football-Scouting/Decision_Quality
pressure radius     : 2.5 m  (>= 2 opp = high pressure)
corridor width (5m) : 5.0 m
pool min minutes    : 135 min


## Section 1 — Corpus construction

H2 evaluates **open-play passes played with the foot**. The H1 pipeline already filtered events to open play and persisted the per-pass enriched table at `data/hull_events_lb.csv` (one row per qualifying pass, with EPV start/end, sender pressure count, geometry tags, etc.). We start from there — no need to re-pull StatsBomb events.

However, H1 retained only what it needed for the line-breaker indices. For H2 we additionally need:

1. **`pass_height`** (Ground / Low / High) — xPass feature #13.
2. **Detailed pass outcome** — H1 stored a binary `pass_successful`; xPass's label distinguishes `Complete` (label = 1) from `Incomplete / Out / Pass Offside / Unknown / Injury Clearance` (label = 0). With the binary flag we are conflating *Out* and *Offside* with *Incomplete*, which is fine for the label but we keep the original outcome to audit edge cases (e.g. unknown).
3. **Receiver position in the frame** — the chosen pass's receiver and *all visible teammates* (for the alternative set, §3). H1 stored `end_x_m` / `end_y_m` (the pass's end location) but not the freeze-frame teammates.
4. **Headers / pass-into-space exclusions** (§4) — H1 already excluded set pieces; we still need to drop headers (body-part) and pass-into-space (`min_dist(end, teammates_in_frame) > 5 m`).

This section first audits what we already have in `hull_events_lb.csv`, then identifies the exact extraction we need to do against the StatsBomb open data.

In [2]:
# load the H1-enriched corpus
lb = pd.read_csv(h1_config.HULL_EVENTS_LB)
print(f'rows                 : {len(lb):,}')
print(f'unique matches       : {lb["match_id"].nunique()}')
print(f'unique players       : {lb["player"].nunique()}')
print(f'pass_successful=True : {lb["pass_successful"].sum():,} ({lb["pass_successful"].mean():.1%})')
print()
print('columns we will reuse for H2 xPass features:')
print('  start position    -> start_x_m, start_y_m')
print('  end position      -> end_x_m,   end_y_m   (chosen pass receiver)')
print('  sender pressure   -> n_close_opp           (= sender_pressure_count_2.5m, feature #8)')
print('  defenders bypass  -> defenders_bypassed    (corridor 5m → close to feature #5)')
print('  EPV at start/end  -> epv_start, epv_end    (for xEPV at chosen pass)')
print('  pass_outcome bin  -> pass_successful       (for xPass label)')
print('  role              -> macro_role            (for within-role percentiles)')
print()
print('what is MISSING and must be re-extracted from StatsBomb 360:')
print('  - pass_height (Ground / Low / High)')
print('  - detailed pass.outcome (Out / Offside / Unknown vs Incomplete)')
print('  - body_part (to drop headers)')
print('  - all visible teammates in the 360 frame -> alternative set')
print('  - all visible opponents in the frame     -> features per teammate (defenders_in_corridor_2m, dist_def_to_receiver, densities, receiver_pressure)')

rows                 : 38,166
unique matches       : 51
unique players       : 357
pass_successful=True : 33,057 (86.6%)

columns we will reuse for H2 xPass features:
  start position    -> start_x_m, start_y_m
  end position      -> end_x_m,   end_y_m   (chosen pass receiver)
  sender pressure   -> n_close_opp           (= sender_pressure_count_2.5m, feature #8)
  defenders bypass  -> defenders_bypassed    (corridor 5m → close to feature #5)
  EPV at start/end  -> epv_start, epv_end    (for xEPV at chosen pass)
  pass_outcome bin  -> pass_successful       (for xPass label)
  role              -> macro_role            (for within-role percentiles)

what is MISSING and must be re-extracted from StatsBomb 360:
  - pass_height (Ground / Low / High)
  - detailed pass.outcome (Out / Offside / Unknown vs Incomplete)
  - body_part (to drop headers)
  - all visible teammates in the 360 frame -> alternative set
  - all visible opponents in the frame     -> features per teammate (defenders_in_co